# Практика · Одноетапні детектори: focal loss і anchor-free

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі дані зошит генерує формулами, а архітектури
> `retinanet_resnet50_fpn` і `fcos_resnet50_fpn` збираються офлайн через
> `weights=None`. Досить `torch`, `torchvision`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **вісімнадцять** маленьких детекторів (шість настройок × три
> зерна). Заміряно: **близько трьох хвилин** на чотирьох ядрах без відеокарти —
> приблизно 9 секунд на одне навчання, 187 секунд на весь зошит.

Що зробимо:

1. порахуємо **дисбаланс** нашої одноетапної голови: скільки позицій позитивні, скільки фонові;
2. подивимось, що з цього виходить, якщо викидати негативні приклади вибіркою **1 : 3**;
3. розберемо **focal loss** по кроках і побудуємо таблицю тлумлення для `p` = 0.9 … 0.1 і γ = 0 … 5;
4. напишемо focal loss **своїми руками** і звіримо з `ops.sigmoid_focal_loss` через `np.allclose`;
5. розкладемо втрату на **фон і предмети** й побачимо, чому легкі приклади забивають градієнт;
6. поміряємо, що робить **ініціалізація зсуву** останнього шару — пару чисел, які вирішують усе;
7. навчимо детектор при **γ = 0, 1, 2, 5** по три зерна й порівняємо `mAP@0.5`;
8. замінимо якірну голову на **anchor-free** і порівняємо параметри, `mAP` і час;
9. перевіримо, що дає **centerness**, і чи справді крайові позиції дають гірші рамки;
10. розберемо `retinanet_resnet50_fpn` і `fcos_resnet50_fpn` і знайдемо, **де саме** вони різняться.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import box_iou, nms, sigmoid_focal_loss
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)
torch.manual_seed(0)

print("torch      ", torch.__version__)
print("numpy      ", np.__version__)
print("потоків    ", torch.get_num_threads())

## 1 · Датасет: ті самі сцени 64×64

Наскрізний приклад блоку не змінюється: полотно 64 на 64 пікселі, від одного до трьох
предметів трьох класів (коло, квадрат, трикутник) радіусом 6-10 пікселів, шум зі
стандартним відхиленням 0.12. Рамка кожного предмета рахується **з його маски**, тому
вона істинна за побудовою.

Набір трохи менший, ніж у [темі 25](../25-yolo/lecture.html) — 240 сцен замість 400.
Причина проста: ми навчатимемо вісімнадцять мереж замість шести, і час треба на щось
витратити.

In [ ]:
SIZE = 64                                  # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3
GRID = 8                                   # карта ознак 8×8
STRIDE = SIZE / GRID                       # крок карти: 8 пікселів на клітинку


def shape_mask(kind, center_x, center_y, radius):
    '''Маска однієї фігури на полотні 64×64.'''
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                   # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                   # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def make_scene(rng, min_objects=1, max_objects=4, min_radius=6, max_radius=11):
    '''Одна сцена: картинка, рамки з масок, мітки класів.'''
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels = [], []
    for _ in range(int(rng.integers(min_objects, max_objects))):
        for _attempt in range(40):
            radius = int(rng.integers(min_radius, max_radius))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            ys, xs = np.nonzero(mask)
            # рамка береться з маски: край + 1, як в угоді COCO
            box = [float(xs.min()), float(ys.min()),
                   float(xs.max() + 1), float(ys.max() + 1)]

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in boxes:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32), np.array(labels, np.int64)


def make_dataset(seed, count):
    rng = np.random.default_rng(seed)
    return [make_scene(rng) for _ in range(count)]


started = time.time()
train_set = make_dataset(42, 240)
test_set = make_dataset(7, 120)
true_box_count = sum(len(scene[1]) for scene in test_set)

# істина в зручному вигляді: список пар (рамки, мітки) по сценах
ground_truth = [(torch.from_numpy(scene[1]), torch.from_numpy(scene[2]))
                for scene in test_set]

print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, перевірних %d" % (len(train_set), len(test_set)))
print("істинних рамок у перевірному наборі: %d" % true_box_count)
print("предметів на сцену в середньому: %.2f"
      % (sum(len(s[1]) for s in train_set) / len(train_set)))

## 2 · Дисбаланс: скільки позицій дивиться одноетапна голова

Наша голова стоїть на карті ознак 8 на 8 і в кожній клітинці має **три квадратні
якорі** розміром 12, 18 і 26 пікселів. Розміри взяті під наші предмети: рамка фігури
радіусом 6-10 пікселів має сторону приблизно від 12 до 21 пікселя.

Правило зіставлення — те саме, що в [темі 24](../24-two-stage/lecture.html), лише
пороги мʼякші: якір позитивний при `IoU ≥ 0.5`, негативний при `IoU < 0.4`, між ними
ігнорується. Плюс рятівне правило: найкращий якір кожної істинної рамки позитивний
завжди.

In [ ]:
ANCHOR_SIZES = (12.0, 18.0, 26.0)          # сторони квадратних якорів у пікселях
ANCHOR_COUNT = len(ANCHOR_SIZES)


def build_anchors():
    '''Три квадратні якорі в центрі кожної клітинки карти 8×8.'''
    out = []
    for row in range(GRID):
        for col in range(GRID):
            center_x = (col + 0.5) * STRIDE
            center_y = (row + 0.5) * STRIDE
            for size in ANCHOR_SIZES:
                out.append([center_x - size / 2, center_y - size / 2,
                            center_x + size / 2, center_y + size / 2])
    return torch.tensor(out, dtype=torch.float32)


ANCHORS = build_anchors()


def anchor_targets(boxes, labels, positive_iou=0.5, negative_iou=0.4):
    '''Стан кожного якоря: 1 позитивний, 0 фон, -1 ігнорується.'''
    total = ANCHORS.shape[0]
    state = torch.zeros(total, dtype=torch.long)
    class_id = torch.full((total,), -1, dtype=torch.long)
    matched = torch.zeros(total, 4)
    if len(boxes) == 0:
        return state, class_id, matched

    truth = torch.from_numpy(boxes)
    overlaps = box_iou(ANCHORS, truth)                 # (якорів, істин)
    best_iou, best_truth = overlaps.max(dim=1)
    state[(best_iou >= negative_iou) & (best_iou < positive_iou)] = -1
    state[best_iou >= positive_iou] = 1

    # рятівне правило: найкращий якір кожної істини стає позитивним попри поріг
    for truth_index in range(truth.shape[0]):
        anchor_index = int(overlaps[:, truth_index].argmax().item())
        state[anchor_index] = 1
        best_truth[anchor_index] = truth_index

    positive = state == 1
    class_id[positive] = torch.from_numpy(labels)[best_truth[positive]]
    matched[positive] = truth[best_truth[positive]]
    return state, class_id, matched


positive_total, ignored_total = 0, 0
for _image, boxes, labels in train_set:
    state, _class_id, _matched = anchor_targets(boxes, labels)
    positive_total += int((state == 1).sum())
    ignored_total += int((state == -1).sum())

decisions = ANCHORS.shape[0] * CLASS_COUNT
positive_per_scene = positive_total / len(train_set)

print("якорів на сцену:                     %d = 64 клітинки × %d розміри"
      % (ANCHORS.shape[0], ANCHOR_COUNT))
print("бінарних рішень «клас чи не клас»:   %d = %d якорів × %d класи"
      % (decisions, ANCHORS.shape[0], CLASS_COUNT))
print()
print("позитивних якорів на сцену:          %.2f (%.3f %% від усіх якорів)"
      % (positive_per_scene, 100 * positive_per_scene / ANCHORS.shape[0]))
print("ігнорованих якорів на сцену:         %.2f"
      % (ignored_total / len(train_set)))
print("позитивних рішень:                   %.3f %%"
      % (100 * positive_per_scene / decisions))
print("на один позитивний припадає:         %.0f негативних"
      % ((decisions - positive_per_scene) / positive_per_scene))

### Милиця перша: викидати негативні приклади

До focal loss дисбаланс лікували вибіркою. Faster R-CNN бере збалансований мінібатч
якорів, SSD залишає найважчі негативні у співвідношенні **1 : 3** до позитивних
(це зветься **hard negative mining**, жорсткий добір негативних прикладів).

Порахуємо, чого це коштує на нашій сцені.

In [ ]:
kept_negatives = 3 * positive_per_scene
kept_total = positive_per_scene + kept_negatives

print("рішень на сцені всього:              %d" % decisions)
print("позитивних:                          %.2f" % positive_per_scene)
print("залишається негативних при 1 : 3:    %.2f" % kept_negatives)
print("разом у втраті:                      %.2f з %d" % (kept_total, decisions))
print()
print("викинуто зі сцени:                   %.1f %% усіх рішень"
      % (100 * (decisions - kept_total) / decisions))
print()
print("Тобто на кожному кроці мережа не отримує градієнта майже ні від чого, що є")
print("на картинці. Фон, який вона впевнено вгадала, зникає з навчання зовсім —")
print("а разом із ним і сигнал «так, тут справді нічого немає, тримай цю думку».")

## 3 · Focal loss по кроках

Звичайна крос-ентропія для одного бінарного рішення — це `−log p`, де `p` —
ймовірність, яку модель дала **правильній** відповіді. Якщо модель упевнена й права
(`p` близьке до одиниці), логарифм близький до нуля, і приклад майже нічого не додає
до втрати. «Майже нічого» — ключове слово: не нуль.

Focal loss домножає крос-ентропію на **множник тлумлення** `(1 − p)^γ`:

`FL = −α (1 − p)^γ · log p`

- `p` — упевненість у правильній відповіді, від 0 до 1;
- `γ` (гамма) — показник степеня; при `γ = 0` формула перетворюється на звичайну крос-ентропію;
- `α` (альфа) — стала вага, різна для предметів і фону.

Побудуємо таблицю руками: що focal loss робить із прикладами різної складності.

In [ ]:
confidences = [0.9, 0.7, 0.5, 0.3, 0.1]
gammas = [0.0, 0.5, 1.0, 2.0, 5.0]

header = "   p       CE   " + "".join(["    γ=%-4.1f" % g for g in gammas])
print(header)
print("   " + "-" * (len(header) - 3))
for p in confidences:
    cross_entropy = -math.log(p)
    row = [cross_entropy * (1 - p) ** gamma for gamma in gammas]
    print("  %.1f  %7.4f" % (p, cross_entropy)
          + "".join(["  %8.5f" % value for value in row]))

print()
print("у скільки разів γ=2 тлумить приклад порівняно зі звичайною крос-ентропією:")
for p in confidences:
    cross_entropy = -math.log(p)
    print("   p = %.1f  →  у %7.1f раза тихіше"
          % (p, cross_entropy / (cross_entropy * (1 - p) ** 2)))
print()
print("Легкий приклад (p = 0.9) стає в 100 разів тихішим, важкий (p = 0.1) —")
print("лише в 1.2 раза. Саме це й потрібно: focal loss не викидає легкі приклади,")
print("а стишує їх — плавно, за одним правилом для всіх.")

### Своя реалізація проти бібліотечної

Напишемо формулу двома способами. Перший — буквально з паперу: порахувати `p` через
сигмоїду й узяти логарифм. Другий — той самий вираз, але з готовою стійкою
крос-ентропією `binary_cross_entropy_with_logits`.

Порівняємо обидва з `torchvision.ops.sigmoid_focal_loss`. Різниця буде повчальною.

In [ ]:
def naive_focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    '''Focal loss буквально за формулою: спершу ймовірність, потім логарифм.'''
    p = torch.sigmoid(logits)
    # крос-ентропія: для цілі 1 беремо log p, для цілі 0 — log(1 − p)
    cross_entropy = -(targets * torch.log(p) + (1 - targets) * torch.log(1 - p))
    p_t = p * targets + (1 - p) * (1 - targets)          # упевненість у правильній відповіді
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * cross_entropy * (1 - p_t) ** gamma


def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    '''Те саме, але крос-ентропію рахує стійка функція torch.'''
    p = torch.sigmoid(logits)
    cross_entropy = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * cross_entropy * (1 - p_t) ** gamma


torch.manual_seed(0)
# логіти детектора легко доходять до ±12: саме там формула й ламається
probe_logits = torch.randn(500, 3) * 3
probe_targets = (torch.rand(500, 3) < 0.1).float()

print("реалізація   найбільше розходження з ops.sigmoid_focal_loss   np.allclose")
for name, our_function in (("наївна", naive_focal_loss), ("стійка", focal_loss)):
    worst = 0.0
    all_close = True
    for alpha in (0.25, 0.5, 0.75):
        for gamma in (0.0, 0.5, 1.0, 2.0, 5.0):
            ours = our_function(probe_logits, probe_targets, alpha, gamma)
            library = sigmoid_focal_loss(probe_logits, probe_targets, alpha, gamma,
                                         reduction="none")
            worst = max(worst, float((ours - library).abs().max()))
            all_close = all_close and np.allclose(ours.numpy(), library.numpy(),
                                                  atol=1e-6)
    print("%-11s  %-49.3g  %s" % (name, worst, all_close))

assert np.allclose(focal_loss(probe_logits, probe_targets, 0.25, 2.0).numpy(),
                   sigmoid_focal_loss(probe_logits, probe_targets, 0.25, 2.0).numpy(),
                   atol=1e-6), "наша focal loss розійшлась із бібліотечною!"
print()
print("✅ стійка реалізація збігається з бібліотечною по всіх 15 комбінаціях α і γ")

Різниця між двома нашими реалізаціями — не в математиці, а в арифметиці `float32`.

Коли логіт дорівнює 12, сигмоїда дає `p = 0.9999938`. Число `1 − p` — це `6.1 · 10⁻⁶`,
і при відніманні майже рівних чисел значущі цифри втрачаються. Логарифм від зіпсованого
числа дає похибку близько **0.005** — на кожному з десятків тисяч фонових прикладів.
`binary_cross_entropy_with_logits` рахує те саме, не переходячи через `p`, і похибки
не має.

Далі всюди беремо стійку версію.

## 4 · Дисбаланс живе не в кількості, а у втраті

Тепер найважливіший замір першої половини теми. Візьмімо **свіжу, ще не навчену**
мережу зі звичайною ініціалізацією й порахуймо, скільки втрати дає фон, а скільки —
предмети.

[Тема 25](../25-yolo/lecture.html) робила те саме для YOLO-подібної голови й дістала
92.3 % на фон. У нас голова дивиться на кожен якір окремо, тож подивимось, що вийде тут.

In [ ]:
class Body(nn.Module):
    '''Тіло мережі — те саме, що в темах 23 і 25: 64×64 → карта 8×8.'''

    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.net = nn.Sequential(
            block(1, 16), block(16, 32), block(32, 64),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

    def forward(self, x):
        return self.net(x)


PRIOR = 0.01                                    # бажана ймовірність предмета на старті
BIAS_INIT = -math.log((1 - PRIOR) / PRIOR)      # зсув, що дає саме таку ймовірність


class AnchorDetector(nn.Module):
    '''Якірна голова RetinaNet-типу: клас на кожен якір і зсув до якоря.'''

    def __init__(self, bias_init=BIAS_INIT):
        super().__init__()
        self.body = Body()
        self.classifier = nn.Conv2d(64, ANCHOR_COUNT * CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(64, ANCHOR_COUNT * 4, 3, padding=1)
        # ініціалізація зсуву: див. розділ 5, без неї нічого не працює
        nn.init.constant_(self.classifier.bias, bias_init)

    def forward(self, x):
        features = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1,
                                                                      CLASS_COUNT)
        deltas = self.regressor(features).permute(0, 2, 3, 1).reshape(count, -1, 4)
        return logits, deltas


print("зсув для p = %.2f: %.4f" % (PRIOR, BIAS_INIT))
print("перевірка: сигмоїда від нього = %.4f" % float(torch.sigmoid(torch.tensor(BIAS_INIT))))
print()
probe = AnchorDetector()
print("параметрів у мережі:  %d" % sum(p.numel() for p in probe.parameters()))
print("з них у тілі:         %d" % sum(p.numel() for p in probe.body.parameters()))
print("у голові класифікації: %d" % sum(p.numel() for p in probe.classifier.parameters()))
print("у голові регресії:     %d" % sum(p.numel() for p in probe.regressor.parameters()))

Тепер підготуємо цілі для всього набору — один раз, щоб не рахувати їх у кожному кроці
навчання, — і подивимось на розкладку втрати свіжої мережі.

In [ ]:
def encode_deltas(anchors, truth):
    '''Зсув від якоря до істинної рамки: два зсуви центра й два логарифми розміру.'''
    anchor_w = anchors[:, 2] - anchors[:, 0]
    anchor_h = anchors[:, 3] - anchors[:, 1]
    anchor_cx = anchors[:, 0] + anchor_w / 2
    anchor_cy = anchors[:, 1] + anchor_h / 2
    truth_w = truth[:, 2] - truth[:, 0]
    truth_h = truth[:, 3] - truth[:, 1]
    truth_cx = truth[:, 0] + truth_w / 2
    truth_cy = truth[:, 1] + truth_h / 2
    return torch.stack([(truth_cx - anchor_cx) / anchor_w,
                        (truth_cy - anchor_cy) / anchor_h,
                        torch.log(truth_w / anchor_w),
                        torch.log(truth_h / anchor_h)], dim=1)


def decode_deltas(anchors, deltas):
    '''Зворотне перетворення: зі зсувів назад у рамку xyxy.'''
    anchor_w = anchors[:, 2] - anchors[:, 0]
    anchor_h = anchors[:, 3] - anchors[:, 1]
    anchor_cx = anchors[:, 0] + anchor_w / 2
    anchor_cy = anchors[:, 1] + anchor_h / 2
    center_x = anchor_cx + deltas[:, 0] * anchor_w
    center_y = anchor_cy + deltas[:, 1] * anchor_h
    width = anchor_w * torch.exp(deltas[:, 2].clamp(max=3.0))
    height = anchor_h * torch.exp(deltas[:, 3].clamp(max=3.0))
    return torch.stack([center_x - width / 2, center_y - height / 2,
                        center_x + width / 2, center_y + height / 2], dim=1)


def pack_anchor(dataset):
    '''Готує тензори всього набору: картинки, маски участі й цілі.'''
    images = torch.from_numpy(np.stack([scene[0] for scene in dataset])[:, None])
    total = ANCHORS.shape[0]
    keep = torch.zeros(len(dataset), total, dtype=torch.bool)
    positive = torch.zeros(len(dataset), total, dtype=torch.bool)
    class_target = torch.zeros(len(dataset), total, CLASS_COUNT)
    delta_target = torch.zeros(len(dataset), total, 4)

    for index, (_image, boxes, labels) in enumerate(dataset):
        state, class_id, matched = anchor_targets(boxes, labels)
        keep[index] = state >= 0                       # ігноровані не входять у втрату
        is_positive = state == 1
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            delta_target[index, is_positive] = encode_deltas(ANCHORS[is_positive],
                                                             matched[is_positive])
    return images, keep, positive, class_target, delta_target


train_images, train_keep, train_positive, train_class, train_delta = pack_anchor(train_set)
test_images = torch.from_numpy(np.stack([scene[0] for scene in test_set])[:, None])

print("форма цілі класифікації:", tuple(train_class.shape), "= сцен × якорів × класів")
print("позитивних рішень у наборі: %d із %d"
      % (int(train_positive.sum()) , train_class.numel()))

In [ ]:
# свіжа мережа з НУЛЬОВИМ зсувом — так робить звичайна ініціалізація torch
torch.manual_seed(0)
fresh = AnchorDetector(bias_init=0.0)
batch_images = train_images[:32]
batch_keep = train_keep[:32]
batch_class = train_class[:32]

with torch.no_grad():
    fresh_logits, _deltas = fresh(batch_images)
    per_element = focal_loss(fresh_logits, batch_class, alpha=0.25, gamma=0.0)

mask = batch_keep.unsqueeze(-1)
background_loss = float((per_element * mask * (1 - batch_class)).sum())
object_loss = float((per_element * mask * batch_class).sum())
total_loss = background_loss + object_loss

print("свіжа мережа, нульовий зсув, звичайна крос-ентропія (γ = 0, α = 0.25)")
print("партія з 32 сцен")
print()
print("  фон:      %9.3f  (%.2f %%)" % (background_loss, 100 * background_loss / total_loss))
print("  предмети: %9.3f  (%.2f %%)" % (object_loss, 100 * object_loss / total_loss))
print("  разом:    %9.3f" % total_loss)
print()
print("Для порівняння, у темі 25 при рівних вагах фон давав 92.3 % втрати.")
print("Тут фон дає %.2f %% — бо голова дивиться на кожен якір і кожен клас окремо."
      % (100 * background_loss / total_loss))

## 5 · Ініціалізація зсуву останнього шару

Це та деталь RetinaNet, про яку майже не пишуть, а без неї навчання розвалюється на
першому ж кроці.

Останній шар голови класифікації має зсув (bias). За замовчуванням `torch` ставить
його близько нуля, тож сигмоїда дає ймовірність приблизно `0.5`: **свіжа мережа
вважає, що предмет є скрізь**. Кожен із тисяч фонових якорів голосно помиляється, і
сума їхніх втрат величезна.

Ліки: поставити зсув так, щоб початкова ймовірність предмета дорівнювала `π = 0.01`.

`зсув = −log((1 − π) ÷ π) = −log(99) = −4.5951`

Поміряймо обидва варіанти на одній і тій самій партії.

In [ ]:
def first_step_loss(bias_init, gamma):
    '''Втрата класифікації на самому першому кроці, ще до жодного оновлення ваг.'''
    torch.manual_seed(0)
    model = AnchorDetector(bias_init=bias_init)
    with torch.no_grad():
        logits, _deltas = model(batch_images)
        probability = float(torch.sigmoid(logits).mean())
        per_element = focal_loss(logits, batch_class, alpha=0.25, gamma=gamma)
        background = float((per_element * mask * (1 - batch_class)).sum())
        objects = float((per_element * mask * batch_class).sum())
    # ділимо на кількість позитивних — так нормують втрату в RetinaNet
    positives = max(1, int(train_positive[:32].sum()))
    return ((background + objects) / positives, probability,
            100 * background / (background + objects))


print("зсув        γ    p(предмет) на старті   втрата на позитивний   частка фону")
print("-" * 74)
numbers = {}
for name, bias in (("нульовий", 0.0), ("π = 0.01", BIAS_INIT)):
    for gamma in (0.0, 2.0):
        loss, probability, share = first_step_loss(bias, gamma)
        numbers[(name, gamma)] = loss
        print("%-10s  %.0f    %6.4f                %10.3f            %6.2f %%"
              % (name, gamma, probability, loss, share))

print()
print("крос-ентропія (γ=0):  %8.2f  →  %6.2f   у %.1f раза менше"
      % (numbers[("нульовий", 0.0)], numbers[("π = 0.01", 0.0)],
         numbers[("нульовий", 0.0)] / numbers[("π = 0.01", 0.0)]))
print("focal loss (γ=2):     %8.2f  →  %6.2f   у %.1f раза менше"
      % (numbers[("нульовий", 2.0)], numbers[("π = 0.01", 2.0)],
         numbers[("нульовий", 2.0)] / numbers[("π = 0.01", 2.0)]))
print()
print("Це не про красу числа. Втрата в тридцять разів більша означає градієнт")
print("у тридцять разів більший — і перший же крок оптимізатора зриває ваги")
print("в бік «мовчи скрізь», звідки мережа може й не повернутись.")
print()

# та сама втрата, порахована формулою: усі логіти дорівнюють зсуву, тож p однакове
kept_decisions = int(batch_keep.sum()) * CLASS_COUNT
positive_decisions = int((batch_class * batch_keep.unsqueeze(-1)).sum())
negative_decisions = kept_decisions - positive_decisions
print("у партії з 32 сцен беруть участь %d рішень:" % kept_decisions)
print("   позитивних %d, фонових %d" % (positive_decisions, negative_decisions))
print()


def analytic_first_step(prior, gamma, alpha=0.25):
    '''Втрата першого кроку за формулою, якщо всі логіти дорівнюють зсуву.'''
    background = negative_decisions * (1 - alpha) * prior ** gamma * -math.log(1 - prior)
    objects = positive_decisions * alpha * (1 - prior) ** gamma * -math.log(prior)
    return (background + objects) / positive_decisions, 100 * background / (background + objects)


print("   π      γ    формула   заміряно   частка фону за формулою")
for prior, gamma, measured in ((0.5, 0.0, numbers[("нульовий", 0.0)]),
                               (0.01, 0.0, numbers[("π = 0.01", 0.0)]),
                               (0.5, 2.0, numbers[("нульовий", 2.0)]),
                               (0.01, 2.0, numbers[("π = 0.01", 2.0)])):
    value, share = analytic_first_step(prior, gamma)
    print("  %.2f    %.0f   %8.2f   %8.2f        %6.2f %%"
          % (prior, gamma, value, measured, share))
print()
print("Формула збігається із заміром у межах кількох відсотків: розбіжність від того,")
print("що справжні логіти не однакові, а трохи розкидані навколо зсуву.")

## 6 · Навчання: що робить γ

Тепер головний замір теми. Навчимо той самий детектор при `γ` = 0, 1, 2 і 5, кожен —
**із трьох зерен**, і подивимось на `mAP@0.5`.

`γ = 0` — це звичайна крос-ентропія з ваговим коефіцієнтом `α`, тобто focal loss без
власне focal-частини. Її ми й беремо за точку відліку.

⏱ Ця клітинка йде близько **двох хвилин**: дванадцять навчань приблизно по 9 секунд.

In [ ]:
def anchor_loss(model_output, keep, positive, class_target, delta_target,
                gamma, alpha=0.25):
    '''Втрата якірного детектора: focal по класах + smooth L1 по рамках.'''
    logits, deltas = model_output
    per_element = focal_loss(logits, class_target, alpha, gamma)
    classification = (per_element * keep.unsqueeze(-1)).sum()
    if positive.any():
        regression = F.smooth_l1_loss(deltas[positive], delta_target[positive],
                                      reduction="sum")
    else:
        regression = logits.sum() * 0
    # нормування на кількість позитивних — так робить RetinaNet
    positives = max(1, int(positive.sum().item()))
    return (classification + regression) / positives


def train_anchor(seed, gamma, epochs=12, learning_rate=3e-3):
    '''Навчає якірний детектор з нуля. Зерно керує вагами й порядком партій.'''
    torch.manual_seed(seed)
    model = AnchorDetector()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(train_images))
        for start in range(0, len(train_images), 32):
            batch = order[start:start + 32]
            loss = anchor_loss(model(train_images[batch]), train_keep[batch],
                               train_positive[batch], train_class[batch],
                               train_delta[batch], gamma)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict_anchor(model, images):
    '''Вихід мережі → списки (рамки, оцінки, класи) по сценах.'''
    model.eval()
    logits, deltas = model(images)
    probabilities = torch.sigmoid(logits)
    results = []
    for index in range(images.shape[0]):
        boxes = decode_deltas(ANCHORS, deltas[index])
        score, class_id = probabilities[index].max(dim=1)
        results.append((boxes, score, class_id))
    return results


print("готово: одне навчання — 12 епох по 240 сцен, AdamW, lr = 3e-3")

In [ ]:
def greedy_match(boxes, scores, truth_boxes, iou_threshold):
    '''Жадібне зіставлення за спаданням оцінки — те саме, що в темі 23.'''
    order = torch.argsort(scores, descending=True)
    taken = [False] * len(truth_boxes)
    hits = torch.zeros(len(order))
    if len(truth_boxes) and len(order):
        overlaps = box_iou(boxes[order], truth_boxes)
        for position in range(len(order)):
            best_value, best_index = -1.0, -1
            for truth_index in range(len(truth_boxes)):
                if taken[truth_index]:
                    continue
                if overlaps[position, truth_index].item() > best_value:
                    best_value = overlaps[position, truth_index].item()
                    best_index = truth_index
            if best_index >= 0 and best_value >= iou_threshold:
                taken[best_index] = True
                hits[position] = 1.0
    return scores[order], hits


def average_precision(scores, hits, truth_count):
    '''AP як площа під огинальною кривої точність-повнота.'''
    order = torch.argsort(scores, descending=True)
    ordered_hits = hits[order]
    running_hits = torch.cumsum(ordered_hits, 0)
    running_misses = torch.cumsum(1 - ordered_hits, 0)
    precision = running_hits / (running_hits + running_misses)
    recall = running_hits / truth_count

    envelope = precision.clone()
    for i in range(len(envelope) - 2, -1, -1):
        envelope[i] = max(envelope[i].item(), envelope[i + 1].item())

    area, previous_recall = 0.0, 0.0
    for i in range(len(envelope)):
        area += (recall[i].item() - previous_recall) * envelope[i].item()
        previous_recall = recall[i].item()
    return area


def mean_average_precision(predictions, iou_threshold=0.5, nms_threshold=0.5,
                           score_floor=1e-3):
    '''mAP при одному порозі IoU: середнє AP по трьох класах.'''
    values = []
    for class_index in range(CLASS_COUNT):
        score_parts, hit_parts, truth_count = [], [], 0
        for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                        ground_truth):
            chosen = (scores >= score_floor) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            truth_count += len(this_truth)
            ordered, hits = greedy_match(picked_boxes, picked_scores, this_truth,
                                         iou_threshold)
            score_parts.append(ordered)
            hit_parts.append(hits)
        values.append(average_precision(torch.cat(score_parts), torch.cat(hit_parts),
                                        max(1, truth_count)))
    return float(np.mean(values)), values


gamma_scores = {}
gamma_models = {}
training_seconds = []

for gamma in (0.0, 1.0, 2.0, 5.0):
    values = []
    for seed in (0, 1, 2):
        model, seconds = train_anchor(seed, gamma)
        training_seconds.append(seconds)
        value, _per_class = mean_average_precision(predict_anchor(model, test_images))
        values.append(value)
        if seed == 0:
            gamma_models[gamma] = model
        print("  γ = %.0f  зерно %d:  %4.1f с   mAP@0.5 = %.4f" % (gamma, seed, seconds, value))
    gamma_scores[gamma] = values
    print("  γ = %.0f  середнє %.4f, розкид %.4f"
          % (gamma, float(np.mean(values)), max(values) - min(values)))
    print()

In [ ]:
print("  γ    зерно 0   зерно 1   зерно 2   середнє   розкид")
print("  " + "-" * 52)
for gamma in (0.0, 1.0, 2.0, 5.0):
    values = gamma_scores[gamma]
    print("  %.0f    %.4f    %.4f    %.4f    %.4f    %.4f"
          % (gamma, values[0], values[1], values[2],
             float(np.mean(values)), max(values) - min(values)))

best_gamma = max(gamma_scores, key=lambda g: float(np.mean(gamma_scores[g])))
gain = float(np.mean(gamma_scores[best_gamma])) - float(np.mean(gamma_scores[0.0]))
loss_at_five = float(np.mean(gamma_scores[best_gamma])) - float(np.mean(gamma_scores[5.0]))
biggest_spread = max(max(v) - min(v) for v in gamma_scores.values())

print()
print("найкраще γ = %.0f, середній mAP %.4f"
      % (best_gamma, np.mean(gamma_scores[best_gamma])))
print("виграш над звичайною крос-ентропією (γ = 0):  %+.4f" % gain)
print("програш при надто великому γ = 5:             %+.4f" % -loss_at_five)
print("найбільший розкид усередині однієї настройки:  %.4f" % biggest_spread)
print()

# порівняння середніх при великому розкиді ненадійне, тому дивимось на крайні зерна
worst_best = min(gamma_scores[best_gamma])
best_zero = max(gamma_scores[0.0])
print("найгірше зерно γ = %.0f:  %.4f" % (best_gamma, worst_best))
print("найкраще зерно γ = 0:   %.4f" % best_zero)
if worst_best > best_zero:
    print("→ УСІ три зерна γ = %.0f кращі за УСІ три зерна γ = 0." % best_gamma)
    print("  Це сильніше твердження, ніж різниця середніх: воно не залежить від")
    print("  того, наскільки великий розкид усередині кожної настройки.")
else:
    print("→ Купи зерен перетинаються: різницю на трьох зернах не впіймано.")
print()
print("Виграш γ = 2 над γ = 0 (%+.4f) сам по собі менший за розкид γ = 0 (%.4f)"
      % (gain, max(gamma_scores[0.0]) - min(gamma_scores[0.0])))
print("і без порівняння крайніх зерен його не варто було б називати справжнім.")
print("А от провал при γ = 5 (%+.4f) більший за будь-який розкид у таблиці —"
      % -loss_at_five)
print("тут сумніватись нема в чому: занадто сильне тлумлення глушить і корисні")
print("приклади теж.")
print()
print("Окремо подивись на розкид: при γ = 0 він %.4f, при γ = 2 — %.4f."
      % (max(gamma_scores[0.0]) - min(gamma_scores[0.0]),
         max(gamma_scores[2.0]) - min(gamma_scores[2.0])))
print("Focal loss не лише піднімає середнє, а й робить навчання передбачуванішим.")

### Хто саме дає втрату після навчання

Тепер повернімось до заміру з розділу 4, але на **навченій** мережі. Розділимо всі
рішення на три купи:

- **позитивні** — там, де предмет справді є;
- **важкі негативні** — фон, у якому мережа підозрює предмет (`p ≥ 0.1`);
- **легкі негативні** — фон, у якому мережа впевнена (`p < 0.1`).

І подивимось, скільки втрати дає кожна купа при різних `γ`.

In [ ]:
reference = gamma_models[2.0]
reference.eval()
with torch.no_grad():
    logits, _deltas = reference(batch_images)
    probability = torch.sigmoid(logits)

keep_mask = batch_keep.unsqueeze(-1).expand_as(logits)
positive_mask = (batch_class > 0.5) & keep_mask
negative_mask = (batch_class < 0.5) & keep_mask
easy_mask = negative_mask & (probability < 0.1)
hard_mask = negative_mask & (probability >= 0.1)

print("на партії з 32 сцен:")
print("  позитивних рішень:        %6d" % int(positive_mask.sum()))
print("  важких негативних:        %6d" % int(hard_mask.sum()))
print("  легких негативних:        %6d" % int(easy_mask.sum()))
print()
print("   γ    позитивні        важкі негат.     легкі негат.")
print("  " + "-" * 56)
contribution = {}
for gamma in (0.0, 0.5, 1.0, 2.0, 5.0):
    per_element = focal_loss(logits, batch_class, alpha=0.25, gamma=gamma)
    parts = [float((per_element * positive_mask).sum()),
             float((per_element * hard_mask).sum()),
             float((per_element * easy_mask).sum())]
    total = sum(parts)
    contribution[gamma] = [100 * p / total for p in parts]
    print("  %.1f   %7.3f (%4.1f %%)  %7.3f (%4.1f %%)  %7.3f (%4.1f %%)"
          % (gamma, parts[0], 100 * parts[0] / total,
             parts[1], 100 * parts[1] / total,
             parts[2], 100 * parts[2] / total))

print()
print("Легких негативних у %d разів більше за позитивні, і при звичайній"
      % (int(easy_mask.sum()) // max(1, int(positive_mask.sum()))))
print("крос-ентропії вони дають %.1f %% усієї втрати — більше за всі предмети разом."
      % contribution[0.0][2])
print("При γ = 2 їхня частка падає до %.1f %%, а частка предметів росте з %.1f %% до %.1f %%."
      % (contribution[2.0][2], contribution[0.0][0], contribution[2.0][0]))

## 7 · Anchor-free: прибираємо якорі

Якірна голова тримає три припущення, які треба вгадувати наперед: **скільки** якорів,
**яких розмірів** і **яких пропорцій**, плюс два пороги IoU для зіставлення.
[Тема 24](../24-two-stage/lecture.html) показала, чим це коштує: з девʼяти варіантів
якоря на її сцені працювали два, а 28.3 % істинних рамок не мали жодного якоря з
IoU понад 0.7.

Anchor-free голова у стилі **FCOS** робить інакше. У кожній позиції карти ознак вона
передбачає **чотири відстані** — до лівого, верхнього, правого й нижнього краю рамки.
Заготовок немає взагалі, вгадувати нічого.

Правило зіставлення теж простіше: позиція позитивна, якщо її центр **усередині**
істинної рамки. Якщо всередині кількох — береться рамка з найменшою площею.

In [ ]:
def build_points():
    '''Центр кожної клітинки карти 8×8 у пікселях вхідного зображення.'''
    out = []
    for row in range(GRID):
        for col in range(GRID):
            out.append([(col + 0.5) * STRIDE, (row + 0.5) * STRIDE])
    return torch.tensor(out, dtype=torch.float32)


POINTS = build_points()


def free_targets(boxes, labels):
    '''Ціль anchor-free: позиція позитивна, якщо вона всередині рамки.'''
    total = POINTS.shape[0]
    positive = torch.zeros(total, dtype=torch.bool)
    class_id = torch.full((total,), -1, dtype=torch.long)
    distances = torch.zeros(total, 4)
    if len(boxes) == 0:
        return positive, class_id, distances

    truth = torch.from_numpy(boxes)
    areas = (truth[:, 2] - truth[:, 0]) * (truth[:, 3] - truth[:, 1])
    point_x = POINTS[:, 0:1]
    point_y = POINTS[:, 1:2]
    left = point_x - truth[:, 0]
    top = point_y - truth[:, 1]
    right = truth[:, 2] - point_x
    bottom = truth[:, 3] - point_y

    inside = (left > 0) & (top > 0) & (right > 0) & (bottom > 0)
    # серед рамок, що містять точку, беремо найменшу за площею
    area_or_infinity = torch.where(inside, areas.expand_as(inside),
                                   torch.full_like(inside, float("inf"),
                                                   dtype=torch.float32))
    smallest_area, chosen = area_or_infinity.min(dim=1)
    positive = torch.isfinite(smallest_area)

    index = torch.arange(total)
    stacked = torch.stack([left[index, chosen], top[index, chosen],
                           right[index, chosen], bottom[index, chosen]], dim=1)
    distances[positive] = stacked[positive] / STRIDE     # ділимо на крок карти
    class_id[positive] = torch.from_numpy(labels)[chosen[positive]]
    return positive, class_id, distances


def centerness_target(distances):
    '''Наскільки позиція близька до центра рамки: 1 у центрі, 0 на краю.'''
    left, top, right, bottom = distances.unbind(dim=1)
    horizontal = torch.min(left, right) / torch.max(left, right).clamp(min=1e-6)
    vertical = torch.min(top, bottom) / torch.max(top, bottom).clamp(min=1e-6)
    return torch.sqrt((horizontal * vertical).clamp(min=0))


# приклад руками: рамка 20×20 з центром у (30, 30), позиція в (28, 30)
example_box = torch.tensor([[20.0, 20.0, 40.0, 40.0]])
for offset in (0, 2, 4, 6, 8):
    left, right = 10.0 + offset, 10.0 - offset
    top, bottom = 10.0, 10.0
    value = math.sqrt((min(left, right) / max(left, right))
                      * (min(top, bottom) / max(top, bottom)))
    print("зсув %d px від центра: l = %4.1f, r = %4.1f  →  centerness = %.3f"
          % (offset, left, right, value))

In [ ]:
class FreeDetector(nn.Module):
    '''Anchor-free голова: класи, чотири відстані й (за бажанням) centerness.'''

    def __init__(self, use_centerness=True, bias_init=BIAS_INIT):
        super().__init__()
        self.body = Body()
        self.classifier = nn.Conv2d(64, CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(64, 4, 3, padding=1)
        self.centerness = nn.Conv2d(64, 1, 3, padding=1) if use_centerness else None
        nn.init.constant_(self.classifier.bias, bias_init)

    def forward(self, x):
        features = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1,
                                                                      CLASS_COUNT)
        # відстані невідʼємні за змістом, тому пропускаємо їх крізь ReLU
        distances = F.relu(self.regressor(features)).permute(0, 2, 3, 1).reshape(count,
                                                                                 -1, 4)
        if self.centerness is None:
            return logits, distances, None
        center = self.centerness(features).permute(0, 2, 3, 1).reshape(count, -1)
        return logits, distances, center


def pack_free(dataset):
    images = torch.from_numpy(np.stack([scene[0] for scene in dataset])[:, None])
    total = POINTS.shape[0]
    positive = torch.zeros(len(dataset), total, dtype=torch.bool)
    class_target = torch.zeros(len(dataset), total, CLASS_COUNT)
    distance_target = torch.zeros(len(dataset), total, 4)
    center_target = torch.zeros(len(dataset), total)

    for index, (_image, boxes, labels) in enumerate(dataset):
        is_positive, class_id, distances = free_targets(boxes, labels)
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            distance_target[index, is_positive] = distances[is_positive]
            center_target[index, is_positive] = centerness_target(distances[is_positive])
    return images, positive, class_target, distance_target, center_target


(free_images, free_positive, free_class,
 free_distance, free_center) = pack_free(train_set)

probe_free = FreeDetector()
body_parameters = sum(p.numel() for p in Body().parameters())

print("позицій на сцену:                  %d" % POINTS.shape[0])
print("позитивних позицій на сцену:       %.2f (%.2f %%)"
      % (free_positive.sum().item() / len(train_set),
         100 * free_positive.sum().item() / free_positive.numel()))
print()
print("параметрів у голові якірній:       %d"
      % (sum(p.numel() for p in AnchorDetector().parameters()) - body_parameters))
print("параметрів у голові anchor-free:   %d"
      % (sum(p.numel() for p in probe_free.parameters()) - body_parameters))
print("те саме без гілки centerness:      %d"
      % (sum(p.numel() for p in FreeDetector(use_centerness=False).parameters())
         - body_parameters))
print()
print("чисел на позицію:")
print("   якірна                  %2d = %d якорі × (%d класи + 4 зсуви)"
      % (ANCHOR_COUNT * (CLASS_COUNT + 4), ANCHOR_COUNT, CLASS_COUNT))
print("   anchor-free             %2d = %d класи + 4 відстані"
      % (CLASS_COUNT + 4, CLASS_COUNT))
print("   anchor-free з centerness %2d = те саме плюс одне число" % (CLASS_COUNT + 5))

Навчаємо anchor-free детектор — теж три зерна, теж `γ = 2`. Спершу **без** гілки
centerness, щоб порівняння з якірною головою було чистим: різниця лише в тому, що
предбачається в кожній позиції.

⏱ Приблизно **тридцять секунд**.

In [ ]:
def free_loss(model_output, positive, class_target, distance_target, center_target,
              gamma, alpha=0.25):
    '''Втрата anchor-free детектора.'''
    logits, distances, center = model_output
    classification = focal_loss(logits, class_target, alpha, gamma).sum()
    if positive.any():
        regression = F.smooth_l1_loss(distances[positive], distance_target[positive],
                                      reduction="sum")
    else:
        regression = logits.sum() * 0
    center_loss = logits.sum() * 0
    if center is not None and positive.any():
        center_loss = F.binary_cross_entropy_with_logits(
            center[positive], center_target[positive], reduction="sum")
    positives = max(1, int(positive.sum().item()))
    return (classification + regression + center_loss) / positives


def train_free(seed, gamma=2.0, use_centerness=True, epochs=12, learning_rate=3e-3):
    torch.manual_seed(seed)
    model = FreeDetector(use_centerness=use_centerness)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(free_images))
        for start in range(0, len(free_images), 32):
            batch = order[start:start + 32]
            loss = free_loss(model(free_images[batch]), free_positive[batch],
                             free_class[batch], free_distance[batch],
                             free_center[batch], gamma)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict_free(model, images, use_centerness_score=True):
    '''Чотири відстані → рамка. Оцінка — клас або корінь із класу × centerness.'''
    model.eval()
    logits, distances, center = model(images)
    probabilities = torch.sigmoid(logits)
    results = []
    for index in range(images.shape[0]):
        d = distances[index] * STRIDE                # назад у пікселі
        boxes = torch.stack([POINTS[:, 0] - d[:, 0], POINTS[:, 1] - d[:, 1],
                             POINTS[:, 0] + d[:, 2], POINTS[:, 1] + d[:, 3]], dim=1)
        score, class_id = probabilities[index].max(dim=1)
        if center is not None and use_centerness_score:
            score = torch.sqrt(score * torch.sigmoid(center[index]))
        results.append((boxes, score, class_id))
    return results


free_plain_scores = []
for seed in (0, 1, 2):
    model, seconds = train_free(seed, use_centerness=False)
    training_seconds.append(seconds)
    value, _per_class = mean_average_precision(predict_free(model, test_images))
    free_plain_scores.append(value)
    if seed == 0:
        free_plain_model = model
    print("  anchor-free без centerness, зерно %d:  %4.1f с   mAP@0.5 = %.4f"
          % (seed, seconds, value))
print("  середнє %.4f, розкид %.4f"
      % (float(np.mean(free_plain_scores)),
         max(free_plain_scores) - min(free_plain_scores)))

In [ ]:
anchor_scores = gamma_scores[2.0]
anchor_model = gamma_models[2.0]

anchor_parameters = sum(p.numel() for p in anchor_model.parameters())
free_parameters = sum(p.numel() for p in free_plain_model.parameters())


def forward_milliseconds(model, images):
    model.eval()
    one_scene = images[:1]
    with torch.no_grad():
        for _warmup in range(5):
            model(one_scene)
        samples = []
        for _ in range(40):
            started = time.time()
            model(one_scene)
            samples.append(time.time() - started)
    return 1000 * float(np.median(samples))


anchor_ms = forward_milliseconds(anchor_model, test_images)
free_ms = forward_milliseconds(free_plain_model, test_images)

print("                      якірна        anchor-free")
print("  " + "-" * 48)
print("  mAP@0.5 (сер.)      %.4f        %.4f"
      % (float(np.mean(anchor_scores)), float(np.mean(free_plain_scores))))
print("  розкид              %.4f        %.4f"
      % (max(anchor_scores) - min(anchor_scores),
         max(free_plain_scores) - min(free_plain_scores)))
print("  параметрів усього   %6d        %6d" % (anchor_parameters, free_parameters))
print("  з них у голові      %6d        %6d"
      % (anchor_parameters - body_parameters, free_parameters - body_parameters))
print("  прохід, мс          %6.2f        %6.2f" % (anchor_ms, free_ms))
print()
print("  гіперпараметрів     розміри якорів (%d), пропорції,   немає жодного"
      % ANCHOR_COUNT)
print("                      два пороги IoU")
print()
print("Голова схудла у %.1f раза, а mAP навіть виріс на %+.4f."
      % ((anchor_parameters - body_parameters) / (free_parameters - body_parameters),
         float(np.mean(free_plain_scores)) - float(np.mean(anchor_scores))))
print("Час проходу майже не змінився — його визначає тіло мережі, а не голова.")

## 8 · Centerness: чи справді крайові позиції гірші

FCOS додає до голови третю гілку — **centerness**, окреме число «наскільки ця позиція
близька до центра рамки». Формула бере відношення менших відстаней до більших:

`centerness = √( min(l,r)/max(l,r) · min(t,b)/max(t,b) )`

У центрі рамки всі чотири відстані рівні, і значення дорівнює одиниці. На краю одна з
відстаней прямує до нуля, і значення падає. Під час передбачення оцінка рамки
множиться на centerness, щоб крайові рамки опустились у списку.

Спершу перевіримо саму передумову: **чи справді позиція біля краю дає гіршу рамку?**
Візьмемо навчений детектор без centerness і подивимось на IoU рамок, побудованих із
позицій різної «центральності».

In [ ]:
free_plain_model.eval()
with torch.no_grad():
    _logits, test_distances, _center = free_plain_model(test_images)

centerness_values, iou_values = [], []
for index, (_image, boxes, labels) in enumerate(test_set):
    if len(boxes) == 0:
        continue
    is_positive, _class_id, distances = free_targets(boxes, labels)
    if not is_positive.any():
        continue
    d = test_distances[index] * STRIDE
    predicted = torch.stack([POINTS[:, 0] - d[:, 0], POINTS[:, 1] - d[:, 1],
                             POINTS[:, 0] + d[:, 2], POINTS[:, 1] + d[:, 3]], dim=1)
    truth = torch.from_numpy(boxes)
    best_iou = box_iou(predicted[is_positive], truth).max(dim=1).values
    centerness_values.append(centerness_target(distances[is_positive]))
    iou_values.append(best_iou)

centerness_values = torch.cat(centerness_values).numpy()
iou_values = torch.cat(iou_values).numpy()

print("позитивних позицій у перевірному наборі: %d" % len(centerness_values))
print("кореляція Пірсона centerness ↔ IoU рамки: %.3f"
      % np.corrcoef(centerness_values, iou_values)[0, 1])
print()
print("  centerness позиції   позицій   середній IoU рамки")
print("  " + "-" * 48)
edges = [0.0, 0.2, 0.4, 0.6, 0.8, 1.001]
for low, high in zip(edges[:-1], edges[1:]):
    chosen = (centerness_values >= low) & (centerness_values < high)
    if chosen.sum():
        print("  %.1f … %.1f              %5d          %.3f"
              % (low, min(high, 1.0), chosen.sum(), iou_values[chosen].mean()))
print()
print("Передумова підтвердилась: що далі позиція від центра, то гірша рамка.")
print("Різниця між крайніми купами — %.3f IoU."
      % (iou_values[centerness_values >= 0.8].mean()
         - iou_values[centerness_values < 0.2].mean()))

Передумова вірна. Тепер перевіримо, чи дає це виграш насправді: навчимо той самий
детектор **із** гілкою centerness і порівняємо три способи рахувати оцінку рамки.

⏱ Ще **тридцять секунд**.

In [ ]:
with_centerness, without_factor = [], []
for seed in (0, 1, 2):
    model, seconds = train_free(seed, use_centerness=True)
    training_seconds.append(seconds)
    with_score, _p = mean_average_precision(predict_free(model, test_images, True))
    plain_score, _p = mean_average_precision(predict_free(model, test_images, False))
    with_centerness.append(with_score)
    without_factor.append(plain_score)
    if seed == 0:
        centerness_model = model
    print("  зерно %d:  %4.1f с   оцінка з centerness %.4f   без множника %.4f"
          % (seed, seconds, with_score, plain_score))

print()
print("  спосіб                                mAP@0.5   розкид")
print("  " + "-" * 52)
print("  гілки немає взагалі                   %.4f    %.4f"
      % (float(np.mean(free_plain_scores)),
         max(free_plain_scores) - min(free_plain_scores)))
print("  гілка є, оцінка без множника          %.4f    %.4f"
      % (float(np.mean(without_factor)), max(without_factor) - min(without_factor)))
print("  гілка є, оцінка × centerness          %.4f    %.4f"
      % (float(np.mean(with_centerness)),
         max(with_centerness) - min(with_centerness)))
print()

centerness_model.eval()
with torch.no_grad():
    _l, _d, learned_center = centerness_model(test_images)
learned, truth_iou = [], []
for index, (_image, boxes, labels) in enumerate(test_set):
    if len(boxes) == 0:
        continue
    is_positive, _class_id, distances = free_targets(boxes, labels)
    if not is_positive.any():
        continue
    learned.append(torch.sigmoid(learned_center[index][is_positive]))
    truth_iou.append(centerness_target(distances[is_positive]))
learned = torch.cat(learned).numpy()
truth_iou = torch.cat(truth_iou).numpy()
print("навчена гілка проти своєї ж цілі: кореляція %.3f"
      % np.corrcoef(learned, truth_iou)[0, 1])
print()
print("ЧЕСНИЙ ВИСНОВОК. На наших сценах centerness не дає нічого — навпаки, забирає.")
print("Причина видна в числах вище: наші предмети маленькі (12-21 px при кроці карти")
print("8 px), тож на один предмет припадає лише %.1f позицій, і крайових серед них мало."
      % (free_positive.sum().item() / sum(len(scene[1]) for scene in train_set)))
print("Ідея правильна, а виграш стає видним на великих предметах, де позицій")
print("усередині рамки десятки й багато з них справді на краю.")

## 9 · Зіставлення без якорів: як обирають відповідальну позицію

Наше правило «центр позиції всередині рамки» — найпростіше з можливих. Справжній FCOS
додає до нього ще дві умови, і обидві можна прочитати прямо з коду `torchvision`:

1. **центральна вибірка** (center sampling): позиція має бути не просто всередині
   рамки, а в межах `radius × крок` від її центра; за замовчуванням `radius = 1.5`;
2. **діапазон масштабів**: кожен рівень піраміди відповідає лише за предмети свого
   розміру. Позиція бере предмет, тільки якщо найбільша з чотирьох відстаней
   потрапляє у вікно `(крок × 4, крок × 8)`.

Друга умова і є те, як anchor-free детектор розводить великі й малі предмети по рівнях
піраміди — без жодних якорів. Порахуємо ці вікна для стандартної піраміди.

In [ ]:
strides = [8, 16, 32, 64, 128]
level_names = ["P3", "P4", "P5", "P6", "P7"]

print("рівень   крок   вікно розмірів (найбільша відстань до краю)")
print("-" * 58)
for name, stride in zip(level_names, strides):
    low = 0 if stride == strides[0] else stride * 4
    high = "∞" if stride == strides[-1] else "%d" % (stride * 8)
    print("  %s     %3d    від %4d до %s" % (name, stride, low, high))

print()
print("Тобто предмет із половиною сторони до 64 px іде на P3, від 64 до 128 — на P4,")
print("і так далі. Один предмет — один рівень, конкуренції між рівнями немає.")
print()
print("Скільки позицій дає сама піраміда на кадрі 800×1216:")
print()
print("рівень   карта        позицій   якорів RetinaNet (×9)")
print("-" * 54)
total_positions = 0
for name, stride in zip(level_names, strides):
    rows = math.ceil(800 / stride)
    columns = math.ceil(1216 / stride)
    total_positions += rows * columns
    print("  %s     %3d×%-4d   %8d   %10d"
          % (name, rows, columns, rows * columns, rows * columns * 9))
print("-" * 54)
print("  разом                %8d   %10d" % (total_positions, total_positions * 9))
print()
print("FCOS дивиться на ті самі %d позицій, але має по одному передбаченню"
      % total_positions)
print("на позицію замість девʼяти — і саме звідси береться різниця в голові.")
print()
print("У нас піраміди немає: одна карта 8×8 з кроком %d. Наші предмети мають" % int(STRIDE))
print("півсторони 6-11 px, тож усі вони потрапили б на найдрібніший рівень.")

## 10 · Справжні моделі: RetinaNet проти FCOS

Обидві архітектури є в `torchvision` і будуються офлайн. Подивимось, **де саме** вони
різняться — відповідь буде дуже конкретною.

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn, fcos_resnet50_fpn

# weights=None і weights_backbone=None: архітектура будується, нічого не тягнеться
retina = retinanet_resnet50_fpn(weights=None, weights_backbone=None)
fcos = fcos_resnet50_fpn(weights=None, weights_backbone=None)


def count_parameters(module):
    return sum(p.numel() for p in module.parameters())


print("retinanet_resnet50_fpn   %10d" % count_parameters(retina))
print("fcos_resnet50_fpn        %10d" % count_parameters(fcos))
print("різниця                  %10d" % (count_parameters(retina) - count_parameters(fcos)))
print()
print("частина                       RetinaNet        FCOS")
print("-" * 56)
print("хребет (ResNet-50 + FPN)      %10d  %10d"
      % (count_parameters(retina.backbone), count_parameters(fcos.backbone)))
print("голова класифікації           %10d  %10d"
      % (count_parameters(retina.head.classification_head),
         count_parameters(fcos.head.classification_head)))
print("голова регресії               %10d  %10d"
      % (count_parameters(retina.head.regression_head),
         count_parameters(fcos.head.regression_head)))
print()
print("якорів на позицію:   RetinaNet %s   FCOS %s"
      % (retina.anchor_generator.num_anchors_per_location()[0],
         fcos.anchor_generator.num_anchors_per_location()[0]))
print()
print("останній шар класифікації:")
print("  RetinaNet  %s" % retina.head.classification_head.cls_logits)
print("  FCOS       %s" % fcos.head.classification_head.cls_logits)
print("  819 = 9 якорів × 91 клас проти 91 = 1 позиція × 91 клас")
print()
print("гілки регресійної голови FCOS: %s"
      % [name for name, _m in fcos.head.regression_head.named_children()])
print("  bbox_ctrness — це і є centerness, окрема згортка на одне число")

In [ ]:
retina_bias = float(retina.head.classification_head.cls_logits.bias[0].detach())
fcos_bias = float(fcos.head.classification_head.cls_logits.bias[0].detach())

print("зсув останнього шару класифікації, як його ставить torchvision:")
print("  RetinaNet  %.4f  →  p = %.4f" % (retina_bias, 1 / (1 + math.exp(-retina_bias))))
print("  FCOS       %.4f  →  p = %.4f" % (fcos_bias, 1 / (1 + math.exp(-fcos_bias))))
print("  наш        %.4f  →  p = %.4f" % (BIAS_INIT, PRIOR))
print()
print("Це та сама формула −log((1 − π) ÷ π) при π = 0.01, і вона зашита")
print("в бібліотеку. Деталь, про яку не пишуть у переказах статті, живе")
print("в коді кожної реалізації.")

## 11 · Подивимось на передбачення очима

In [ ]:
predictions = predict_free(free_plain_model, test_images[:3])

figure, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for axis, (image, truth_boxes, _labels), (boxes, scores, _class_id) in zip(
        axes, test_set[:3], predictions):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    for box in truth_boxes:
        axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                     fill=False, edgecolor="#0f766e", linewidth=1.6))
    chosen = scores >= 0.3
    picked_boxes, picked_scores = boxes[chosen], scores[chosen]
    if len(picked_boxes):
        keep = nms(picked_boxes, picked_scores, 0.5)
        for position in keep.tolist():
            box = picked_boxes[position]
            axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0],
                                         box[3] - box[1], fill=False,
                                         edgecolor="#c2185b", linewidth=1.2,
                                         linestyle="--"))
    axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout()
plt.show()
print("бірюзові суцільні — істина, малинові пунктирні — anchor-free детектор після NMS")

## 12 · Підсумок зошита

In [ ]:
print("Що поміряно:")
print()
print("  бінарних рішень на сцену                 %d" % decisions)
print("  позитивних із них                        %.2f (%.3f %%)"
      % (positive_per_scene, 100 * positive_per_scene / decisions))
print("  на один позитивний негативних            %.0f"
      % ((decisions - positive_per_scene) / positive_per_scene))
print("  вибірка 1 : 3 викидає                    %.1f %% рішень сцени"
      % (100 * (decisions - kept_total) / decisions))
print()
print("  фон у втраті свіжої мережі               %.2f %%"
      % (100 * background_loss / total_loss))
print("  легкі негативні у втраті, γ = 0          %.1f %%" % contribution[0.0][2])
print("  легкі негативні у втраті, γ = 2          %.1f %%" % contribution[2.0][2])
print("  предмети у втраті, γ = 0 → γ = 2         %.1f %% → %.1f %%"
      % (contribution[0.0][0], contribution[2.0][0]))
print()
print("  втрата на першому кроці, зсув 0          %.2f" % numbers[("нульовий", 0.0)])
print("  те саме зі зсувом π = 0.01               %.2f" % numbers[("π = 0.01", 0.0)])
print("  focal, зсув 0 → π = 0.01                 %.2f → %.2f"
      % (numbers[("нульовий", 2.0)], numbers[("π = 0.01", 2.0)]))
print()
print("  mAP@0.5 при γ = 0                        %.4f (розкид %.4f)"
      % (float(np.mean(gamma_scores[0.0])),
         max(gamma_scores[0.0]) - min(gamma_scores[0.0])))
print("  mAP@0.5 при γ = 1                        %.4f" % float(np.mean(gamma_scores[1.0])))
print("  mAP@0.5 при γ = 2                        %.4f (розкид %.4f)"
      % (float(np.mean(gamma_scores[2.0])),
         max(gamma_scores[2.0]) - min(gamma_scores[2.0])))
print("  mAP@0.5 при γ = 5                        %.4f" % float(np.mean(gamma_scores[5.0])))
print()
print("  anchor-free mAP@0.5                      %.4f" % float(np.mean(free_plain_scores)))
print("  параметрів у голові, якірна → free       %d → %d"
      % (anchor_parameters - body_parameters, free_parameters - body_parameters))
print("  centerness: без гілки → з множником      %.4f → %.4f"
      % (float(np.mean(free_plain_scores)), float(np.mean(with_centerness))))
print()
print("  retinanet_resnet50_fpn                   %d параметрів" % count_parameters(retina))
print("  fcos_resnet50_fpn                        %d параметрів" % count_parameters(fcos))
print("  хребет у них однаковий                   %d" % count_parameters(retina.backbone))
print()
print("  навчань усього                           %d" % len(training_seconds))
print("  середній час одного навчання             %.1f с" % float(np.mean(training_seconds)))
print("  разом на навчання                        %.0f с" % sum(training_seconds))

## Завдання

### 🟢 Рівень 1 — База

Побудуй графік focal loss від `p` для `γ` = 0, 0.5, 1, 2, 5 на одних осях. Познач на
ньому точку `p = 0.9` і підпиши, у скільки разів `γ = 2` тлумить цей приклад.

**Зроблено, якщо:** графік побудовано, і ти можеш назвати `γ`, при якому приклад із
`p = 0.7` стає тихішим за приклад із `p = 0.1` при `γ = 0`.

### 🟡 Рівень 2 — Плюс

Прожени `first_step_loss` для `π` = 0.5, 0.1, 0.05, 0.01, 0.001 і побудуй графік
«втрата на першому кроці проти `π`». Поясни словами, чому крива не спадає нескінченно:
що починає рости, коли `π` стає надто малим.

**Зроблено, якщо:** графік побудовано, мінімум знайдено чисельно, і пояснення спирається
на два доданки втрати окремо (фон і предмети), а не на загальну суму.

### 🔴 Рівень 3 — Виклик

Реалізуй свою anchor-free голову з нуля (можна взяти нашу за зразок, але напиши
зіставлення сам — із центральною вибіркою `radius = 1.5` з розділу 9) і порівняй
`mAP@0.5` та `mAP@0.5:0.95` з якірною головою **при однаковому бюджеті**: те саме тіло,
та сама кількість епох, ті самі три зерна.

**Зроблено, якщо:** чотири числа надруковано в одній таблиці разом із розкидом по
зернах, і ти можеш сказати, чи різниця більша за розкид — і що саме змінила центральна
вибірка порівняно з простим правилом «усередині рамки».